# Mini-TP 2 — Actividad entregable (Sesión 2)

**Expón los metadatos de tu modelo por GraphQL y compáralo con REST.**
Individual · entrega esta semana. Completa las celdas marcadas con `# TODO`.

## Consigna
1. Define un **esquema GraphQL** (Strawberry) con un tipo `Model` (`name`, `version`, `metrics`).
2. Una **query** que devuelva las métricas/experimentos de tu modelo (de MLflow local o simuladas).
3. Pruébalo desde **GraphiQL** y desde un **cliente Python**.
4. **Compara** la misma lectura contra tu endpoint **REST** de la Sesión 1 (llamadas y datos) y anota la diferencia.

**Se evalúa:** que corra de punta a punta; que el esquema tipe entrada y salida; que la query pida solo lo necesario; y la reflexión REST vs GraphQL.
**Opcional (+):** que el resolver lea el **linaje** desde Neo4j (ver `graphql_neo4j_lineage.ipynb`).

## 0. Requisitos
```bash
uv pip install strawberry-graphql fastapi "uvicorn[standard]" requests
```

In [ ]:
# !uv pip install strawberry-graphql fastapi "uvicorn[standard]" requests

## 1. Los metadatos de TU modelo

Reemplaza estos datos por los de tu modelo (o leelos de MLflow).

In [ ]:
# TODO: pon aquí los metadatos reales de tu modelo (o deja los de ejemplo)
MI_MODELO = {
    "name": "TODO_nombre",
    "version": 1,
    "metrics": {"auc": 0.0, "accuracy": 0.0, "f1": 0.0},  # TODO
}

## 2. El esquema GraphQL

Completa el tipo `Model` y el resolver que devuelve sus métricas.

In [ ]:
import strawberry
from typing import Optional
from fastapi import FastAPI
from strawberry.fastapi import GraphQLRouter

@strawberry.type
class Metrics:
    auc: float
    accuracy: float
    f1: float

@strawberry.type
class Model:
    name: str
    version: int

    @strawberry.field
    def metrics(self) -> Metrics:
        m = MI_MODELO["metrics"]
        # TODO: si lees de MLflow, trae las métricas reales aquí
        return Metrics(auc=m["auc"], accuracy=m["accuracy"], f1=m["f1"])

@strawberry.type
class Query:
    @strawberry.field
    def model(self) -> Model:
        return Model(name=MI_MODELO["name"], version=MI_MODELO["version"])

schema = strawberry.Schema(query=Query)
app = FastAPI(title="Mini-TP 2 - metadatos por GraphQL")
app.include_router(GraphQLRouter(schema), prefix="/graphql")
print("Esquema listo")

## 3. Levantar la API (en segundo plano) y consultar

In [ ]:
import threading, time, uvicorn
def _run():
    uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8010, log_level="warning")).run()
threading.Thread(target=_run, daemon=True).start(); time.sleep(2)

import requests
query = "{ model { name version metrics { auc accuracy f1 } } }"
r = requests.post("http://127.0.0.1:8010/graphql", json={"query": query})
print(r.status_code, r.json())

## 4. Comparación con REST  `# TODO`

Sirve el mismo modelo por REST (puedes reusar tu API de la Sesión 1) y cuenta cuántas llamadas y cuántos bytes necesitas para armar la misma vista. Compara con GraphQL.

> Referencia: `rest_vs_graphql.ipynb` hace exactamente esta comparación con código.

In [ ]:
# TODO: tu comparación REST vs GraphQL (llamadas y bytes) y una breve conclusión escrita.

## 5. (Opcional +) Linaje desde Neo4j
Si te animas, haz que un resolver devuelva el **linaje** de tu modelo leyendo de Neo4j. Base: `graphql_neo4j_lineage.ipynb`.

---
### Qué entregar
* Este notebook corriendo de punta a punta.
* La query GraphQL y su salida.
* La comparación REST vs GraphQL (números + conclusión).
* En el README de tu repo, una línea sobre qué diferencia notaste.